# TCC Nível Guaíba - Exploração de Dados

## Introdução

* Neste notebook, iremos explorar os dados obtidos das estações hidrometereológicas obtidas através da API do Hidroweb, um serviço do Sistema Nacional de Informações sobre Recursos Hídricos (SNIRH) disponibilizado pela Agência Nacional de Águas (ANA). A mesma contém informações mais de 23 mil estações de monitoramento cadastradas, com dados em especial de níveis fluviais e climatologia;

* O objetivo deste trabalho é a comparação de modelos predição do nível da água no Rio Guaíba, para isso serão necessárias as Séries Históricas referentes à este nível da água, com estas informações sendo complementadas por outros dados metereológicos, como chuva, chuva acumulada e temperatura, todas também obtidas do HidroWeb;

* Dado o foco do trabalho, as estações coletadas são referentes à Bacia Hidrográfica do Guaíba, uma das principais do Rio Grande do Sul. Foram escolhidas estações representantes dos principais rios afluentes do Rio Guaíba, os rios Taquari, Caí, Gravataí, Sinos e Jacuí, vizando obter a maior cobertura e qualidade de dados possível; 

* Swagger da API: https://www.ana.gov.br/hidrowebservice/swagger-ui/index.html#/

## API HidroWeb

* A lista das possíveis estações foi coletada do Endpoint https://www.ana.gov.br/hidrowebservice/EstacoesTelemetricas/HidroInventarioEstacoes/v1;

* Das estações coletadas, fazem parte da Bacia Hidrográfica do Guaíba aqueles Rios com código:
    * Bacia = 'ATLÂNTICO, TRECHO SUDESTE'
    * Sub_Bacia = 'LAGOA DOS PATOS'
    * Bacia_Codigo = '8'
    * Sub_Bacia_Codigo = '87'

* Os códigos (Rio_Codigo) referentes aos rios afluentes selecionados são:
    * Rio Guaíba = '87200000'
    * Rio Taquari = '86001000'
    * Rio Caí = '87220000'
    * Rio Gravataí = '87240000'
    * Rio Sinos = '87230000'
    * Rio Jacuí = '85001000'

* Nestes rios, o números de estações totaliza 258. Porém, muitas destas estações não estão mais em funcionamento, com somente o registro histórico presente na base de dados, além daquelas que não são telemétricas, cujo uso não seria apropriado dado o objetivo de predições de curto prazo que requerem dados em tempo real. Logo, foi necessário filtrar as estações nestes rios para somente aquelas com os parâmetros Operando IN ('1') e Tipo_Estacao_Telemetrica IN ('1'), totalizando 43 possíveis estações;
    * Essas estações ainda foram filtradas além, dado que muitas apresentam uma utilização estratégica para uma função específica e não apresentam dados hidrometereológicos como esperado. Por exemplo, estações energéticas, utilizadas em barragens, ou estações piezométricas, utilizadas para medir a pressão em aquíferos;  
    * `Rio_Codigo IN ('87200000', '86001000', '87220000', '87240000', '87230000', '85001000')  and Operando IN ('1') and Tipo_Estacao_Telemetrica IN ('1');`

* As séries históricas destas estações tiveram seus dados analizados, focando na presença dos dados necessários, a qualidade dos mesmos, a presença de lacunas, o tamanho da série histórica e a representatividade de cada afluente;

* Como alvo das predições foi escolhida uma estação no Rio Guaíba, a estação 'CAIS MAUÁ C6', visto que a mesma foi utilizada como referência do nível do Guaíba durante as enchentes de 2024 e serviu como forma de monitoramento da crise. Contudo, devido a enchente no Cais da Mauá, os equipamentos de medição da estação foram danificados no dia 2 de maio de 2024, o que levou à instalação emergencial da estação 'USINA DO GASÔMETRO (CHEIA 2024)', que continua a ser usada até hoje. Logo, os dados desta estação foram considerados como a continuação daquelas do Caís da Mauá; 

* As séries detalhadas de cada estação foram obtidas deste endpoint: https://www.ana.gov.br/hidrowebservice/EstacoesTelemetricas/HidroinfoanaSerieTelemetricaDetalhada/v1;

* No fim do processo de coleta de dados, depois de muitos testes manuais foram escolhidas a seguintes estações com Estação_Nome ('codigoestacao'):
    * Rio Guaíba = CAIS MAUÁ C6 ('87450004'), USINA DO GASÔMETRO (CHEIA 2024) ('87444000')
    * Rio Taquari = MUÇUM ('86510000'), ENCANTADO ('86720000')
    * Rio Caí = LINHA GONZAGA ('87150000'), BARCA DO CAÍ ('87170000')
    * Rio Gravataí = PASSO DAS CANOAS - AUXILIAR ('87399000')
    * Rio Sinos = SÃO LEOPOLDO ('87382000'), CAMPO BOM ('87380000')
    * Rio Jacuí = RIO PARDO ('85900000')

* Cada uma destas estações apresenta uma data diferente como início de coleta dos dados. Para que o tamanho das séries fossem iguais, foi escolhido a data de '2018-08-01' como data de corte, dado que este é o primeiro período na qual todas apresentam registros significativos. Logo, o coletado foi referente à 6 anos de dados telemétricos;
    * Além disso, cada estação apresenta um registo à cada 15 minutos, totalizando X observações;
    
* Os dados disponibilizados na API apresentavam um limite no período de coleta de 30 dias, logo o período de coleta foi dividido em chunks de 30 dias de concatenados no final do processo;

## Bibliotecas

In [ ]:
# ML Libraries;
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
from pyod.models.pca import PCA
from pyod.models.ecod import ECOD

# Ploting Libraries;
import os
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ydata_profiling import ProfileReport

# Internal imports;
from source_database.db_handler import DBConnection
from util import convert_to_float, STATIONS_COLS, AGG_DICT, START_DATE, END_DATE
from source_frontend.make_station_comparison import make_station_comparison
from source_database.data_creator import collect_all_stations, clean_dataframe, fill_gaps, aggregate_data, outlier_removal, feature_imputation, melt_dataframe

## Visualização Dados

In [ ]:
# Initialize the Connection;
db = DBConnection()

# Query data;
query = {
    'cai_1':        'SELECT * FROM station_cai_1',
    'cai_2':        'SELECT * FROM station_cai_2',
    'gravatai_1':   'SELECT * FROM station_gravatai_1',
    'guaiba_1':     'SELECT * FROM station_guaiba_1',
    'guaiba_2':     'SELECT * FROM station_guaiba_2',
    'jacui_1':      'SELECT * FROM station_jacui_1',
    'sinos_1':      'SELECT * FROM station_sinos_1',
    'sinos_2':      'SELECT * FROM station_sinos_2',
    'taquari_1':    'SELECT * FROM station_taquari_1',
    'taquari_2':    'SELECT * FROM station_taquari_2',
    }
dataframe = db.run(query=query)

# Merge guaiba_1 and guaiba_2 into a single dataframe;
guaiba_merged = pd.concat([dataframe['guaiba_1'], dataframe['guaiba_2']])
guaiba_merged['codigoestacao'] = '87450004'

# Update the dictionary with the merged guaiba and remove guaiba_2;
dataframe['guaiba_1'] = guaiba_merged
dataframe.pop('guaiba_2')

# Concatenate all station dataframes;
df = pd.concat(list(dataframe.values()))
df.head()

In [ ]:
# Convert values and cut the dataframe to a time range where most data is available;
df_cleaned = clean_dataframe(df=df)

In [ ]:
# # Automated data profiling;
# profile = ProfileReport(df_cleaned, title="Profiling Report")
# profile.to_notebook_iframe()

### Valores faltantes

* Pode-se ver que certos atributos estão consistentemente faltando do dataset. Em especial, todos aqueles relacionados à Cota que não o 'Cota_Adotada'. Estes atributos podem ser considerados secundários na análise e serão tratadados de tal forma;
    * Na análise à seguir, veremos que todos os 'Cota_*' são altamente relacionados com os valores de 'Cota_Adotada' e, como o mesmo é o nosso alvo de análises, usaremos valores de 'Cota_Manual', 'Cota_Display' e 'Cota_Sensor' para preencher valores faltantes de 'Cota_Adotada';

* Além disso, 'Pressao_Atmosferica' e 'Temperatura_Agua' infelizmente apresentam muitos valores faltantes para serem de qualquer uso nessa análise e logo serão desconsiderados daqui pra frente;

* Por fim, vale notar que estes valores são a média para todas as estações e, quando tratando cada uma de forma individual, a quantidade de valores faltantes varia bastante. Buscou-se coletar dados de estações que estivessem com o mínimo de valores faltantes possível;

In [ ]:
# Percentage of Null values;
100*df_cleaned.isnull().sum()/df_cleaned.shape[0]

### Correlação

* A análise de correlação mostra que as colunas **'Cota_Manual'**, **'Cota_Sensor'** e **'Cota_Display'** têm alta correlação com o target **'Cota_Adotada'**;
    * A variável **'Cota_Manual'** apresenta mais de **99% de valores nulos**, e seus poucos valores não nulos ocorrem majoritariamente quando **'Cota_Adotada'** está ausente. Isso indica que essas medições manuais servem para substituir as automáticas durante períodos de manutenção ou falha do sensor;
    * As variáveis **'Cota_Sensor'** e **'Cota_Display'** são, na maioria dos casos, idênticas à **'Cota_Adotada'**, o que é consistente com o fato de que as leituras são predominantemente obtidas de sensores automáticos;

* A variável **Vazao_Adotada** apresenta alta correlação com **Cota_Adotada**, o que é consistente com o comportamento hidrodinâmico: o aumento da vazão ($m^3/s$) em um leito fixo eleva o nível da água até atingir as áreas de várzea adjacentes. Essa variável será fundamental para a predição, dada sua forte relação com o target e baixa correlação com outras variáveis — indicando que traz informação complementar relevante ao modelo;

* No entanto, há uma alta taxa de valores ausentes (>15%), cuja magnitude varia entre estações. A estação alvo (**Guaíba**) não possui registros de vazão, o que limita seu uso direto nessa localidade;

In [ ]:
%matplotlib inline
a = df_cleaned.reset_index()
columns = [col for col in df_cleaned.columns if not col.endswith('_Status')]
columns = list(set(columns) - set(['Pressao_Atmosferica', 'Temperatura_Agua']))
a = a[columns]

plt.figure(figsize=(25, 15))
sns.heatmap(
    data=a.corr(),
    annot=True,
    fmt=".1f",
    annot_kws={"size": 10},
)
plt.xticks(rotation=45, ha='right')  # tilt x-axis labels for readability
plt.yticks(rotation=0)
plt.tight_layout()
plt.show("png")

### Comparação entre as estações

* Ao comparar os dados de nível da água entre as estações, nota-se um dos principais problemas dos dados hidrometeorológicos do HidroWeb: **gaps de dados**. Todas as estações apresentam falhas, em diferentes intensidades, com períodos sem registros;

* Na estação **RIO PARDO ('85900000')** evidencia-se bem esse problema. Seus gaps são extensos, com o maior de **nov/2020** a **jul/2021**. Nesse intervalo, há registros pontuais, mas sem a frequência regular de **15 minutos** observada no restante do dataset;
    * Nem todos os gaps apresentam registros esparsos. Em alguns períodos, os dados simplesmente não existem;

In [ ]:
df_cleaned.head()

In [ ]:
# make_station_comparison(df=df_cleaned)

![AA](/home/juju/Documents/Nivel_TCC/data/comparação_nível_estações.png)

## Preenchimento de falhas

* Para tratar as falhas de frequência presentes nas estações, adotaram-se duas abordagens:
    * **Interpolação linear** para gaps pequenos (≤ 2h, ou até 8 intervalos de 15 min). Esse limite é arbitrário e pode ser reajustado;
    * **Imputação iterativa** para gaps grandes, usando `IterativeImputer` com `ExtraTreesRegressor`, robusto a relações não lineares;

* Para a interpolação linear, foi necessário criar uma timeline continua de 15 minutos para todas as estações, para que estes valores que não estão presente, mas que não são fáceis de vizualizar, pudessem ser preenchidos;

* Para a variável 'Cota_Adotada' também foi realizada uma imputação manual dos valores faltantes usando os valores de **'Cota_Manual'** e **'Cota_Sensor'**;

* Com a linha temporal completa para cada estação, o número de valores faltantes aumenta consideravelmente. Abaixo a quantidade de valores faltantes de **'Cota_Adotada'** por estação:

In [ ]:
# Fill the data gaps;
df_filled, missing_values = fill_gaps(df=df_cleaned, max_fill_steps=8)

In [ ]:
# Create grouped bar chart with two bars per station;
plot = sns.catplot(
    data=missing_values,
    kind='bar',
    x='station_id',
    y='missing_percentage',
    hue='period',
    height=6,
    aspect=2
)

# Add labels to both groups of bars;
for ax in plot.axes.flat:
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', fontsize=10);

plot.set_xlabels('Estação')
plot.set_ylabels('Dados faltantes de Cota_Adotada (%)')
plot.fig.suptitle('Dados faltantes - Antes e depois do gap filling')
plot.set_xticklabels(rotation=45, ha='right')
plt.tight_layout()
plt.show()

* O processo de imputação com `IterativeImputer` será realizado posteriormente, após o tratamento e substituição dos outliers, que seguirão a mesma abordagem.


### Colunas de Status

* Cada coluna de informação é acompanhada por uma coluna de Status, referente ao estado da coleta destes valores pelos sensores da estação;

* Foi seguida a convenção adotada pelo SNIRH, com a inclusão de uma flag para valores que foram imputados e/ou valores faltantes;

    * Status Codes Quality Control Convention: 0=Normal, 1=Suspeito, 2=Ruim, 3=Muito Ruim, 4=Preenchido/Faltante, 5=Outlier;

## Agregação de dados

* A frequência original de 15 minutos à cada registro dos dados do HidroWeb não é ideal para realizar predições tanto de longo prazo, como de curto prazo. Em ambos os casos, seria necessário um horizonte de predição muito grande para realizar qualquer predição relevante temporalmente.

* Logo, foi necessário criar um método para agrupar os dados com uma frequência qualquer;

In [ ]:
# Aggregate the data to the desired frequency;
df_agg = aggregate_data(df=df_filled, frequency='h')

In [ ]:
df_agg.head()

In [ ]:
make_station_comparison(df=df_agg)

![AAA](/home/juju/Documents/Nivel_TCC/data/comparação_agg.png)

* Agregar os dados desta forma permite visualizar uma particularidade dos dados que antes não era visível com a frequência natural de 15 minutos, a presença de outliers. Estes dados claramente fora da escala padrão do resto do dataset estão presentes na frequência original, mas são mais visíveis quando a quantidade de pontos é reduzida;

## Visualização Outliers

In [ ]:
# Prepare data;
index_cols = ['date', 'station_id']
status_cols = [col for col in df.columns if col.endswith('_status')]
non_feature_cols = index_cols + status_cols
feature_cols = list(set(df.columns.unique()) - set(non_feature_cols))

In [ ]:
def outlier_visualization(df, df_features_clean, original_indices, feature_cols,
                          ecod_predictions, ecod_scores, ecod_threshold,
                          pca_predictions, pca_scores, pca_threshold,
                          combined_predictions):
    """
    Create visualizations for outlier detection results;

    Parameters:
        df (pd.DataFrame): Original dataframe;
        df_features_clean (pd.DataFrame): Cleaned features used for detection;
        original_indices (pd.Index): Indices of cleaned data in original df;
        feature_cols (list): List of feature column names;
        ecod_predictions (np.array): ECOD outlier predictions (0=normal, 1=outlier);
        ecod_scores (np.array): ECOD outlier scores;
        ecod_threshold (float): ECOD decision threshold;
        pca_predictions (np.array): PCA outlier predictions;
        pca_scores (np.array): PCA outlier scores;
        pca_threshold (float): PCA decision threshold;
        combined_predictions (np.array): Combined outlier predictions;
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.decomposition import PCA as sklearn_PCA
    
    print("\n" + "-"*80)
    print("CREATING VISUALIZATIONS")
    print("-"*80)
    
    sns.set_style("whitegrid")
    plt.rcParams['figure.facecolor'] = 'white'
    
    # 1. Outlier Scores Distribution;
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    axes[0].hist(ecod_scores[ecod_predictions == 0], bins=50, alpha=0.7, label='Normal', color='blue')
    axes[0].hist(ecod_scores[ecod_predictions == 1], bins=50, alpha=0.7, label='Outlier', color='red')
    axes[0].axvline(ecod_threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold: {ecod_threshold:.3f}')
    axes[0].set_xlabel('Outlier Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('ECOD Outlier Scores Distribution', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(pca_scores[pca_predictions == 0], bins=50, alpha=0.7, label='Normal', color='blue')
    axes[1].hist(pca_scores[pca_predictions == 1], bins=50, alpha=0.7, label='Outlier', color='red')
    axes[1].axvline(pca_threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold: {pca_threshold:.3f}')
    axes[1].set_xlabel('Outlier Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('PCA Outlier Scores Distribution', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_scores_distribution.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_scores_distribution.png")
    plt.close()
    
    # 2. PCA 2D Projection;
    pca_2d = sklearn_PCA(n_components=2)
    features_2d = pca_2d.fit_transform(df_features_clean)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for idx, (predictions, title) in enumerate([(ecod_predictions, 'ECOD'),
                                                  (pca_predictions, 'PCA'),
                                                  (combined_predictions, 'Combined')]):
        axes[idx].scatter(features_2d[predictions == 0, 0], features_2d[predictions == 0, 1],
                         c='blue', alpha=0.5, s=10, label='Normal')
        axes[idx].scatter(features_2d[predictions == 1, 0], features_2d[predictions == 1, 1],
                         c='red', alpha=0.8, s=30, label='Outlier', edgecolors='black', linewidths=0.5)
        axes[idx].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
        axes[idx].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
        axes[idx].set_title(f'{title} Outliers in PCA Space', fontsize=14, fontweight='bold')
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_pca_projection.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_pca_projection.png")
    plt.close()
    
    # 3. Feature-wise boxplots;
    n_features = len(feature_cols)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
    axes = axes.flatten() if n_features > 1 else [axes]
    
    for idx, feature in enumerate(feature_cols):
        normal_data = df_features_clean.loc[combined_predictions == 0, feature]
        outlier_data = df_features_clean.loc[combined_predictions == 1, feature]
        bp = axes[idx].boxplot([normal_data, outlier_data], labels=['Normal', 'Outlier'],
                               patch_artist=True, widths=0.6)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][1].set_facecolor('lightcoral')
        axes[idx].set_ylabel('Value', fontsize=10)
        axes[idx].set_title(f'{feature}', fontsize=11, fontweight='bold')
        axes[idx].grid(True, alpha=0.3, axis='y')
    
    for idx in range(n_features, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Feature Distribution: Normal vs Outliers', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_feature_boxplots.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_feature_boxplots.png")
    plt.close()
    
    # 4. Time series with outliers;
    if 'date' in df.columns:
        df_viz = df.loc[original_indices].copy()
        df_viz['combined_outlier'] = combined_predictions
        
        fig, axes = plt.subplots(len(feature_cols), 1, figsize=(16, len(feature_cols) * 3))
        if len(feature_cols) == 1:
            axes = [axes]
        
        for idx, feature in enumerate(feature_cols):
            axes[idx].plot(df_viz['date'], df_viz[feature], color='gray', alpha=0.5, linewidth=0.5, label='Normal')
            outlier_mask = df_viz['combined_outlier'] == 1
            if outlier_mask.sum() > 0:
                axes[idx].scatter(df_viz.loc[outlier_mask, 'date'], df_viz.loc[outlier_mask, feature],
                                 color='red', s=20, alpha=0.8, label='Outlier', zorder=5)
            axes[idx].set_ylabel(feature, fontsize=10)
            axes[idx].set_title(f'{feature} - Time Series with Outliers', fontsize=11, fontweight='bold')
            axes[idx].grid(True, alpha=0.3)
            if idx == 0:
                axes[idx].legend(fontsize=9)
            if idx < len(feature_cols) - 1:
                axes[idx].set_xticklabels([])
        
        axes[-1].set_xlabel('Date', fontsize=12)
        plt.suptitle('Time Series Analysis - Outliers Highlighted', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_timeseries.png', dpi=300, bbox_inches='tight')
        print("Saved: outlier_timeseries.png")
        plt.close()
    
    # 5. Feature correlation with outliers;
    fig, ax = plt.subplots(figsize=(10, 8))
    df_corr = df_features_clean.copy()
    df_corr['is_outlier'] = combined_predictions
    correlation_with_outlier = df_corr.corr()['is_outlier'].drop('is_outlier').sort_values(ascending=False)
    
    colors = ['red' if x > 0 else 'blue' for x in correlation_with_outlier.values]
    ax.barh(range(len(correlation_with_outlier)), correlation_with_outlier.values, color=colors, alpha=0.7)
    ax.set_yticks(range(len(correlation_with_outlier)))
    ax.set_yticklabels(correlation_with_outlier.index, fontsize=10)
    ax.set_xlabel('Correlation with Outlier Status', fontsize=12)
    ax.set_title('Feature Correlation with Outlier Detection', fontsize=14, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_feature_correlation.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_feature_correlation.png")
    plt.close()
    
    print("\n" + "="*80)
    print("OUTLIER DETECTION COMPLETE - All visualizations saved")
    print("="*80 + "\n")

# Generate visualizations;
outlier_visualization(
    df=df,
    df_features_clean=df_features,
    original_indices=original_indices,
    feature_cols=feature_cols,
    ecod_predictions=ecod_predictions,
    ecod_scores=ecod_scores,
    ecod_threshold=ecod_detector.threshold_,
    pca_predictions=pca_predictions,
    pca_scores=pca_scores,
    pca_threshold=pca_detector.threshold_,
    combined_predictions=combined_predictions
)